In [1]:
import sys
from pathlib import Path
# Ensure project root is on sys.path so 'utils' is importable
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# nb_img_08 — Label Propagation on CLIP Embeddings

Uses LabelSpreading over k-NN graph built on CLIP embeddings to propagate labels from seed set (5%) to unlabeled set (95%) with no model training.

## Section 1 — Load data

In [2]:
import numpy as np
import pandas as pd
import json
from sklearn.metrics import f1_score
from utils.image_config import IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES, RESULTS_DIR
from utils.label_propagation import run_label_propagation

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

seed_mask = manifest["is_seed"].to_numpy().astype(bool)
seed_embs = embeddings[seed_mask]
seed_labels = manifest.loc[seed_mask, "label_idx"].to_numpy()
unlabeled_embs = embeddings[~seed_mask]

print(f"Seed: {seed_mask.sum()}  Unlabeled: {(~seed_mask).sum()}")

Seed: 275  Unlabeled: 5225


## Section 2 — Run label propagation

In [3]:
pred_unlabeled, confidence_unlabeled = run_label_propagation(
    labeled_embeddings=seed_embs,
    labeled_labels=seed_labels,
    unlabeled_embeddings=unlabeled_embs,
    kernel="knn",
    n_neighbors=15,
)

# Reconstruct full-dataset predictions (seed rows get their true label)
all_pred = np.empty(len(manifest), dtype=int)
all_pred[seed_mask] = seed_labels
all_pred[~seed_mask] = pred_unlabeled

all_conf = np.empty(len(manifest), dtype=float)
all_conf[seed_mask] = 1.0
all_conf[~seed_mask] = confidence_unlabeled

true_labels = manifest["label_idx"].to_numpy()
label_acc = (all_pred == true_labels).mean()
label_f1 = f1_score(true_labels, all_pred, average="macro", zero_division=0)
print(f"Label ACC: {label_acc:.4f}  F1: {label_f1:.4f}  Coverage: 100%")

Label ACC: 0.9869  F1: 0.9869  Coverage: 100%


## Section 3 — Save results

In [4]:
out_df = manifest[["filepath", "true_label"]].copy()
out_df["predicted_label"] = [IMAGE_CLASS_NAMES[p] for p in all_pred]
out_df["confidence"] = all_conf
RESULTS_DIR.mkdir(exist_ok=True)
out_df.to_csv(RESULTS_DIR / "image_full_labels_label_propagation.csv", index=False)

metrics = {"label_acc": float(label_acc), "label_f1": float(label_f1), "coverage": 1.0}
with open(RESULTS_DIR / "image_metrics_label_propagation.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'label_acc': 0.986909090909091, 'label_f1': 0.986899554744519, 'coverage': 1.0}
